In [ ]:
import pandas as pd

class ProcessadorDePedidos:

    def __init__(self, caminho_arquivo):
        self.caminho_arquivo = caminho_arquivo
        self.dados = pd.read_excel(caminho_arquivo)
        self.filiais_processadas = {}
        self.condicoes_pedido = {}
        import warnings
        # Desativa todos os warnings
        warnings.filterwarnings('ignore')

    def encontrar_primeiros_dois_nao_numericos_consecutivos(self, df, coluna):
        for idx in range(len(df[coluna]) - 1):
            try:
                int(df[coluna].iloc[idx])
                primeiro_nao_numerico = False
            except ValueError:
                primeiro_nao_numerico = True

            try:
                int(df[coluna].iloc[idx + 1])
                segundo_nao_numerico = False
            except ValueError:
                segundo_nao_numerico = True

            if primeiro_nao_numerico and segundo_nao_numerico:
                return idx + 1  # Retorna o índice do segundo valor não numérico

        return None

    def remover_nan_coluna_0(self, df):
        # Remove os valores NaN da coluna 0
        df = df.dropna(subset=[df.columns[0]]).reset_index(drop=True)
        return df

    def processar_filiais(self):
        linhas_filial = self.dados[self.dados.iloc[:, 0] == "Filial:"]
        indices_filiais = linhas_filial.index

        for indice in indices_filiais:
            nome_filial = self.dados.iloc[indice, 2]
            indice_ajustado = indice - 2
            df_filial = self.dados.iloc[indice_ajustado:,]

            indice_numero_pedido = df_filial[df_filial.iloc[:, 15] == 'Numero Pedido:'].index
            if not indice_numero_pedido.empty:
                numero_pedido = self.dados.iloc[indice_numero_pedido[0], 20]
            else:
                continue

            indice_condicao_pagamento = df_filial[df_filial.iloc[:, 0] == 'Condição de pagamento:'].index
            if not indice_condicao_pagamento.empty:
                condicao_pagamento = self.dados.iloc[indice_condicao_pagamento[0], 4]
            else:
                condicao_pagamento = None

            self.processar_pedido(nome_filial, numero_pedido, condicao_pagamento, df_filial, indice_ajustado)

    def processar_pedido(self, nome_filial, numero_pedido, condicao_pagamento, df_filial, indice_inicial):
        indice_codigo = df_filial[df_filial.iloc[:, 0] == "Código"].index[0]
        inicio_tabela = indice_codigo + 2
        validacao_tabela = self.dados.iloc[inicio_tabela, 0]
        print(validacao_tabela)
        try:
            celula_valida = int(validacao_tabela)
            inicio = inicio_tabela
        except ValueError:
            celula_valida = "PEDIDO VAZIO"

        if isinstance(celula_valida, int):
            fim_coluna = self.encontrar_primeiros_dois_nao_numericos_consecutivos(self.dados.iloc[inicio:,], 'Unnamed: 0')
            fim_coluna = fim_coluna + inicio - 1
            df_pedido = self.dados.iloc[inicio:fim_coluna,]
            self.condicoes_pedido[numero_pedido] = condicao_pagamento

            if nome_filial not in self.filiais_processadas:
                self.filiais_processadas[nome_filial] = {}

            if numero_pedido not in self.filiais_processadas[nome_filial]:
                self.filiais_processadas[nome_filial][numero_pedido] = df_pedido
            else:
                print("Pedido já está lançado.")
                print("Concatenar dataframe.")
                self.filiais_processadas[nome_filial][numero_pedido] = pd.concat([self.filiais_processadas[nome_filial][numero_pedido], df_pedido])

    def obter_dataframe_final(self):
        df_final = pd.DataFrame()
        for nome_filial, pedidos in self.filiais_processadas.items():
            for numero_pedido, dataframe in pedidos.items():
                dataframe['Numero do Pedido'] = numero_pedido
                dataframe['Filial'] = nome_filial
                df_final = pd.concat([df_final, dataframe])#, ignore_index=True)
        # Limpar os dados
        df_final = df_final.dropna(axis=1, how='all').copy()
        

        # Definir as colunas
        colunas = ['Código', 'Descrição', 'Fabricante', 'Und', 'Embalagem', 'Qtd', 'Desc.', 'Vlr Unit', 'Vlr IPI', 'Vlr Total', 'Icms', 'Numero do Pedido', 'Filial']
        df_final.columns = colunas
        
        # Adicionar prazo
        df_final = self.adicionar_prazo(df_final, self.condicoes_pedido)
        
        return df_final

    def adicionar_prazo(self, df, condicoes):
        df['Prazo'] = df.apply(lambda row: condicoes.get(row['Numero do Pedido'], None), axis=1)
        return df

    def exibir_dataframe_final(self):

        df_final = self.obter_dataframe_final()

        df_final = self.remover_nan_coluna_0(df_final)


        return df_final
    
caminho_arquivo = r"C:\Users\Rafael\Downloads\(QUARTO) COTY-GAM  22-02. - sabonete monange - KALUG.xls"
processador = ProcessadorDePedidos(caminho_arquivo)
processador.processar_filiais()
df= processador.exibir_dataframe_final()


In [36]:
df.to_excel(r"C:\Users\Rafael\Downloads\teste-df.xlsx")